In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib
import os

# 1. Load data from DB
engine = create_engine('sqlite:///../telco_churn.db')
df = pd.read_sql("SELECT * FROM raw_churn;", engine)

# 2. Clean up columns
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])


app_features = [
    'tenure', 'MonthlyCharges', 'TotalCharges', 'Contract', 
    'InternetService', 'PaymentMethod', 'PaperlessBilling', 'TechSupport', 'Churn'
]
df_subset = df[app_features].copy()

# 3. Scale only the continuous numerical features before dummy encoding
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
df_subset[num_cols] = scaler.fit_transform(df_subset[num_cols])

# 4. Encode categorical variables into 0s and 1s
df_subset['Churn'] = df_subset['Churn'].map({'Yes': 1, 'No': 0})
df_encoded = pd.get_dummies(df_subset, drop_first=True)

# 5. Separate features from target & split data
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 6. Retrain the model on the correctly scaled data
model_lr = LogisticRegression(max_iter=1000, random_state=42)
model_lr.fit(X_train, y_train)

# 7. Export the clean artifacts
os.makedirs('../models', exist_ok=True)
joblib.dump(model_lr, '../models/winning_churn_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(X.columns.tolist(), '../models/model_features.pkl')

print("Success! Your model and scaler have been saved using clean scaling procedures.")

✅ Success! Your model and scaler have been saved using clean scaling procedures.


In [13]:
# Initialize our competitive models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss')
}

# Loop through, train, and print metrics for each model
for name, model in models.items():
    print(f"=== Training {name} ===")
    
    # Train the model on the scaled training data
    model.fit(X_train_scaled, y_train)
    
    # Predict the test data
    y_pred = model.predict(X_test_scaled)
    
    # Print performance evaluation results
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred))
    print("-" * 50)

=== Training Logistic Regression ===
Accuracy: 0.7939
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1033
           1       0.63      0.53      0.58       374

    accuracy                           0.79      1407
   macro avg       0.74      0.71      0.72      1407
weighted avg       0.79      0.79      0.79      1407

--------------------------------------------------
=== Training Random Forest ===
Accuracy: 0.7790
              precision    recall  f1-score   support

           0       0.83      0.87      0.85      1033
           1       0.60      0.52      0.56       374

    accuracy                           0.78      1407
   macro avg       0.72      0.70      0.70      1407
weighted avg       0.77      0.78      0.77      1407

--------------------------------------------------
=== Training XGBoost ===
Accuracy: 0.7711
              precision    recall  f1-score   support

           0       0.83      0.86      0.85  

In [14]:
# Extract the weights (coefficients) from Logistic Regression
model_lr = models["Logistic Regression"]
coefficients = model_lr.coef_[0]

# Pair them with column names
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': coefficients
}).sort_values(by='Importance', key=abs, ascending=False)

# Display the top 10 most influential features
print("--- Top 10 Features Driving Churn Decisions ---")
print(feature_importance.head(10))

--- Top 10 Features Driving Churn Decisions ---
                            Feature  Importance
0                            tenure   -1.354644
2                      TotalCharges    0.700377
4                 Contract_Two year   -0.645956
5       InternetService_Fiber optic    0.379674
3                 Contract_One year   -0.330627
8    PaymentMethod_Electronic check    0.225804
12                  TechSupport_Yes   -0.198274
11  TechSupport_No internet service   -0.184728
6                InternetService_No   -0.184728
10             PaperlessBilling_Yes    0.183326


In [ ]:
import joblib
import os

# Create a directory to hold saved models if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save the Logistic Regression model and the scaler
joblib.dump(model_lr, '../models/winning_churn_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

# Save the original column names so Streamlit knows the structure
joblib.dump(X.columns.tolist(), '../models/model_features.pkl')

print("Success! Model, Scaler, and Features saved inside the 'models/' folder.")

📦 Success! Model, Scaler, and Features saved inside the 'models/' folder.


In [16]:
import joblib

# Load the exact features list we exported
exported_features = joblib.load('../models/model_features.pkl')

print("--- Notebook Training Features Order ---")
for i, col in enumerate(exported_features):
    print(f"{i}: {col}")

--- Notebook Training Features Order ---
0: tenure
1: MonthlyCharges
2: TotalCharges
3: Contract_One year
4: Contract_Two year
5: InternetService_Fiber optic
6: InternetService_No
7: PaymentMethod_Credit card (automatic)
8: PaymentMethod_Electronic check
9: PaymentMethod_Mailed check
10: PaperlessBilling_Yes
11: TechSupport_No internet service
12: TechSupport_Yes
